# Epidemic curves from a JUNE2 events file (events-only, via `SimulationEvents`)

Same example as `plot_infections.ipynb`, but driven through the `SimulationEvents` facade
instead of hand-rolling the low-level reader calls. Requires `requirements.txt` +
`requirements-render.txt`.

## Parameters
Point `events_path` at any `simulation_events.h5`. The default is an example full-England
run; swap it for the small committed example fixture (see ../README.md) when you want a portable run -- the
structure is identical. `SimulationEvents` accepts a `str` or a `Path`.

In [ ]:
import sys
from pathlib import Path

# Make the repo root importable so `core` resolves (no packaging yet).
repo_root = Path.cwd()
while not (repo_root / "core").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# --- parameters ---
events_path = repo_root.parent / "JUNE2" / "runs" / "run_full_england_modern" / "simulation_events.h5"

print("repo_root :", repo_root)
print("events_path:", events_path, "(exists:", events_path.exists(), ")")

## Bind the run and inspect it
`SimulationEvents` binds one run's events file. `event_types()` lists the event types it
recorded (name + row count) -- the facade owns the `events/` prefix and the path coercion.

The facade is *events-only*, so for the broader file view (the `lookups/` tables,
registries) we still reach for the low-level `inspect_file` -- the facade does not pretend
to own datasets outside `events/`.

In [ ]:
from core.load_data import SimulationEvents
from core.load_data.june_events import inspect_file

sim = SimulationEvents(events_path)

print("event types (name, n_rows):")
for event_type in sim.event_types():
    print(" ", event_type.name, event_type.n_rows)

# Full-file view (lookups, registries) -- outside the facade's events-only scope.
summary = inspect_file(str(events_path))
print("\nall datasets:")
for dataset in summary.datasets:
    print(" ", dataset.path, dataset.n_rows, dataset.dtype)
print("registries:", list(summary.registries))

### Available event types
`event_types()` returns every `events/` type, empties included -- the Consumer filters to
the non-empty ones it wants to plot.

In [ ]:
available_event_types = [
    event_type.name for event_type in sim.event_types() if event_type.n_rows > 0
]
print("available event types:", available_event_types)

## Look at one events list
The light `events` path loads the decoded table (registry codes resolved, no
lookup joins) -- all the curves need is `time`, so the cheap path suffices. The
facade assembles the `events/deaths` path for us.

In [ ]:
events = sim.events("deaths")
events

## Load, aggregate, build curves
For each chosen event type: load only the `time` column via the light `events`
path and group by day to get curves of events/day. The skip-guard keeps the
notebook working on any file -- an event type the run did not record is skipped.

In [ ]:
event_types = ["infections", "deaths", "hospital_admissions"]  # which curves to plot

curves = {}
for event_type in event_types:
    if event_type not in available_event_types:
        print(f"skipping {event_type!r} -- not present in this file")
        continue
    events = sim.events(event_type, columns=['time'])
    curves[event_type] = events.groupby(events["time"].astype(int)).size()
    print(f"{event_type}: {int(curves[event_type].sum())} events")

## Plot

Note: the first day starts with the infection seeds, so there might be an unexpected dip in
the first day or two.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 8), sharex=True)

# Only plot curves the file actually produced (the build loop skips absent types).
if "infections" in curves:
    ax1.plot(curves["infections"].index, curves["infections"], label="infections")
ax1.set_xlabel("time (days)")
ax1.set_ylabel("events per day")
ax1.set_title("Epidemic curves (events-only)")
ax1.set_ylim(bottom=0)

if "deaths" in curves:
    ax2.plot(curves["deaths"].index, curves["deaths"],
             label="deaths", color="black", linestyle="dotted")
if "hospital_admissions" in curves:
    ax2.plot(curves["hospital_admissions"].index, curves["hospital_admissions"],
             label="hospital_admissions", color="forestgreen", linestyle="dashed")
ax2.set_xlabel("time (days)")
ax2.set_ylabel("events per day")
ax2.set_ylim(bottom=0)

ax1.legend()
ax2.legend()
fig.tight_layout()
plt.show()

## Per-geo aggregate (the light geo path)
A curve only needs `time`, but a per-area breakdown (and the map / rate-per-100k path)
needs each event's **geo unit**. `sim.geo_events(type)` is the light feed for that: it
resolves one `geo_unit_id` per event (venue-then-person) by joining *only* that column
from the lookups -- a fraction of the cost of the full `enriched` people+venue join.
`aggregate_events` then bins it into a dense (time x geo) `Aggregate`; `to_long_dataframe`
tidies the slices you want.

In [ ]:
from core.aggregate import aggregate_events, epidemic_curve, to_long_dataframe

days_per_bin = 1.0

if "infections" in available_event_types:
    located = sim.geo_events("infections")            # time, geo_unit_id
    infections_aggregate = aggregate_events(
        located, event_type="infections", days_per_bin=days_per_bin
    )
    # The per-bin total matches the light-events curve above (same events, now geo-keyed).
    print("curve total:", int(epidemic_curve(infections_aggregate)["count"].sum()))
    long_frame = to_long_dataframe(infections_aggregate)
    long_frame.head()

## Roll up to a coarser geo level (leaves the events-only path)

The `Aggregate` above is keyed on whatever geo level this run's lookups happened to
resolve -- often thousands of small units, and not necessarily the same level as another
run. `rollup` re-keys its columns to an ancestor level and sums, so "per region" is one
call rather than a hierarchy walk you write yourself.

The hierarchy comes from the **World file**, so this cell needs `world_reader` installed
and a `world_state.h5` -- exactly the same step outside the events-only path that
rate-per-100k is (README §6). Everything above still runs without one.

Three things to expect:

- **Level names are per run.** `world.geo_levels()` lists this run's own, coarsest first;
  there is no portable `"region"`. Read that output before choosing.
- **Nothing is dropped.** A column with no ancestor at the level you asked for -- the
  `-1` seeds/foreign-travel sentinel, or a unit hanging off a ragged branch -- keeps its
  own column and is warned about once. So the total is conserved, and the result is not
  guaranteed homogeneous in level.
- **Rates need no extra work.** Population is subtree-aggregated in the World file, so a
  region's denominator is already its own entry in the map.

In [ ]:
import pandas as pd

# --- parameter --- point this at the world_state.h5 for the SAME run as events_path.
world_path = events_path.parent / "world_state.h5"
print("world_path:", world_path, "(exists:", world_path.exists(), ")")

if world_path.exists() and "infections" in available_event_types:
    from core.aggregate.rollup import rollup
    from core.load_data.world import load_world

    world = load_world(world_path)
    print("geo levels (coarsest first):", world.geo_levels())

    # One step coarser than the coarsest is a sensible default; name a level from
    # the list above instead if you want a particular one.
    geo_level = world.geo_levels()[1] if len(world.geo_levels()) > 1 else world.geo_levels()[0]
    print(f"rolling up to {geo_level!r}")

    rolled = rollup(infections_aggregate, world.ancestor_by_geo_unit(geo_level))
    print(f"{len(infections_aggregate.geo_unit_ids)} columns -> {len(rolled.geo_unit_ids)}")
    # Coarsening moves counts between columns, it never loses any.
    print("total before:", infections_aggregate.counts.sum(),
          " after:", rolled.counts.sum())

    per_level = pd.DataFrame({
        "geo_unit_id": rolled.geo_unit_ids,
        "infections": rolled.counts.sum(axis=0),
        "peak_rate_per_100k": rolled.rate_per_100k(
            world.population_by_geo_unit()
        ).max(axis=0),
    }).sort_values("infections", ascending=False)
    display(per_level.head(10))
else:
    print("skipping the rollup -- needs a World file for this run (see the note above)")